# Exploring Falco event output formats — JSON, gRPC, and sidekick-forwarding

> last_verified: 2026-09-19 · tool_version: n/a (no version researched this cycle; no externally-verifiable claims made)

## Purpose

This notebook compares the three main ways to consume Falco runtime-security events: structured JSON records, a streaming gRPC-style interface, and forwarding through a sidekick-style relay to downstream consumers. The goal is to help decide which output fits a given consumer — a local log file, a live stream processor, or a fan-out to chat/storage/alerting endpoints.

## When to use

Use JSON output when events need to be stored, searched, or piped into log tooling. Use a streaming interface when a program must react to events as they arrive rather than polling a file. Use a forwarding relay when one event stream must reach several destinations with different formatting or filtering per destination.

## Prerequisites

- A Falco deployment that emits detection events (any environment is fine for this comparison).
- Python 3 with only the standard library (this notebook uses `json` and nothing else).
- Familiarity with the idea of a Falco rule producing an event with a severity-like ranking and a human-readable message.

In [ ]:
import json

# Illustrative event shape only: a simplified stand-in for the fields a
# Falco JSON event carries (ranking, message, which rule fired, when).
# Field names below are illustrative, not a schema claim.
event = {
    "ranking": "Warning",
    "message": "illustrative: shell started inside a container",
    "rule": "illustrative-shell-in-container",
    "time": "illustrative-timestamp",
}

record = json.dumps(event)
print(record)
restored = json.loads(record)
assert restored["rule"] == "illustrative-shell-in-container"
assert set(restored) == {"ranking", "message", "rule", "time"}
print("round-trip OK:", sorted(restored))

## Steps

### 1. JSON as the base interchange shape

JSON is the common denominator: each detection becomes one self-contained record carrying a severity-like ranking, a message, the firing rule, and a timestamp. Because it is plain text with structure, the same record can be appended to a file, shipped to a log aggregator, or parsed by a short script without special client libraries. The cell above shows the round-trip property that makes JSON convenient: serialize, transport, parse, and the fields survive intact.

### 2. Streaming delivery for live reaction

A streaming interface serves events over a long-lived connection instead of writing discrete records for something else to poll. The trade-off is operational: the consumer must stay connected and handle reconnects, but it observes each event once, in order, with minimal delay. This fits automated responders that contain or annotate workloads within seconds of a detection.

### 3. Sidekick-style forwarding for fan-out

A forwarding relay sits between the event stream and the final destinations. It receives each event once and then fans out to any number of outputs — message channels, object storage, alerting endpoints — applying per-output filtering (for example, only high-ranking events reach the paging endpoint while everything reaches storage). The cell below models that routing decision in plain Python.

In [ ]:
HIGH_RANKINGS = {"Critical", "Error"}


def route(event, storage, pager):
    """Fan-out model: everything reaches storage; only high rankings page."""
    storage.append(event)
    if event.get("ranking") in HIGH_RANKINGS:
        pager.append(event)
    return len(storage), len(pager)


storage, pager = [], []
feed = [
    {"ranking": "Notice", "message": "illustrative low-rank event"},
    {"ranking": "Warning", "message": "illustrative mid-rank event"},
    {"ranking": "Critical", "message": "illustrative high-rank event"},
]
for e in feed:
    route(e, storage, pager)
print("stored:", len(storage), "paged:", len(pager))
assert len(storage) == 3
assert len(pager) == 1 and pager[0]["ranking"] == "Critical"

## Verify

Both code cells assert their own behavior: the first checks that a JSON round-trip preserves every field, and the second checks that the fan-out model stores all three illustrative events while paging only the high-ranking one. Re-run the notebook top to bottom; success means both cells complete without an assertion error.

## Common errors

- Treating the illustrative field names above as a guaranteed schema. Real event records carry additional context fields; always inspect an actual record from the deployed setup before writing a parser.
- Forwarding every ranking to the paging endpoint. Without per-output filtering, on-call channels saturate and genuine high-rank detections get missed.
- Polling a stream that expects a persistent consumer. If the consumer reconnects on every event instead of holding the connection, it adds latency and risks gaps.

## References

- `falco/configs/container-drift-detection.yaml` — example rule configuration in this kit.
- `falco/docs/syscall-vs-tracepoint-rules.md` — background on what the events being formatted actually describe.